In [1]:
!apt-get update && apt-get install -y build-essential

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1581 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2533 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:4 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7015 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1311 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3929 kB]
Ign:8 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Ign:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Ign:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Ign:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Ign:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Ign:10 http://arch

To run this, press "*Runtime*" and press "*Run all*" on a **multi-GPU** instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

This notebook is optimized for multi-GPU training. It configures the training arguments to effectively utilize your 4x Tesla T4 GPUs.

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** — a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install accelerate  # Install accelerate for multi-GPU support

### Unsloth

In [3]:
from unsloth import FastLanguageModel
import torch

# To support multi-GPU, we ensure DataParallel handles model if multiple GPUs are available
# Unsloth will still handle patching for memory efficiency

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 1024,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 4. Max memory: 14.609 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


We now add LoRA adapters so we only need to update a small amount of parameters!

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


Unsloth 2026.4.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the `Qwen-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Qwen-3 renders multi turn conversations like below:

```
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>

```
We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [5]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

In [6]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files = {
        "train": "content/train.jsonl",
        "validation": "content/val.jsonl",
        "test": "content/test.jsonl",
    },
)

dataset

DatasetDict({
    train: Dataset({
        features: ['example_id', 'doc_id', 'split', 'domain', 'task', 'summary_type', 'messages', 'text', 'target_format', 'source_doc_id', 'chunk_id', 'token_count_estimate', 'source_spans', 'quality_flags', 'metadata'],
        num_rows: 600
    })
    validation: Dataset({
        features: ['example_id', 'doc_id', 'split', 'domain', 'task', 'summary_type', 'messages', 'text', 'target_format', 'source_doc_id', 'chunk_id', 'token_count_estimate', 'source_spans', 'quality_flags', 'metadata'],
        num_rows: 80
    })
    test: Dataset({
        features: ['example_id', 'doc_id', 'split', 'domain', 'task', 'summary_type', 'messages', 'text', 'target_format', 'source_doc_id', 'chunk_id', 'token_count_estimate', 'source_spans', 'quality_flags', 'metadata'],
        num_rows: 80
    })
})

In [7]:
def normalize_messages(messages):
    normalized = []
    for message in messages:
        content = message["content"]
        if isinstance(content, str):
            content = [{"type": "text", "text": content}]
        normalized.append({
            "role": message["role"],
            "content": content,
        })
    return normalized


Let's see how row 100 looks like!

In [8]:
dataset["train"][100]

{'example_id': 'meritlifeinsuranceco_06_19_2020_ex_10_xiv_master_services_agreement__chunk_003__extract',
 'doc_id': 'meritlifeinsuranceco_06_19_2020_ex_10_xiv_master_services_agreement__chunk_003',
 'split': 'train',
 'domain': 'contract',
 'task': 'extract',
 'summary_type': None,
 'messages': [{'role': 'system',
   'content': 'You are a contract analysis assistant.\nRead the provided agreement and return only valid JSON that matches the contract schema exactly.\nDo not invent clauses, dates, obligations, or penalties.'},
  {'role': 'user',
   'content': "Mode: contract\nTask: extract structured data into the canonical JSON schema.\n\nDocument:\nTo the extent that Contractor incorporates any of Contractor's Information into the Works, Contractor hereby grants to Company a royalty-free, non- exclusive perpetual license (including the right to grant a sublicense) to use, copy, modify, create, derivative version, publicly perform and publicly display such Contractor's Information in con

We now have to apply the chat template for `Qwen-3` onto the conversations, and save it to `text`.

In [9]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, # Removed normalize_messages(convo) as content is already string
            tokenize = False,
            add_generation_prompt = False,
            enable_thinking = False,
        ).removeprefix("<bos>")
        for convo in convos
    ]
    return {"text": texts}

formatted = dataset.map(formatting_prompts_func, batched = True)

train_dataset = formatted["train"].remove_columns(
    [c for c in formatted["train"].column_names if c != "text"]
)
eval_dataset = formatted["validation"].remove_columns(
    [c for c in formatted["validation"].column_names if c != "text"]
)
test_dataset = formatted["test"].remove_columns(
    [c for c in formatted["test"].column_names if c != "text"]
)

Let's see how the chat template did!

In [10]:
train_dataset[0]["text"]

'<|im_start|>system\nYou are a contract analysis assistant.\nRead the provided agreement and return only valid JSON that matches the contract schema exactly.\nDo not invent clauses, dates, obligations, or penalties.<|im_end|>\n<|im_start|>user\nMode: contract\nTask: extract structured data into the canonical JSON schema.\n\nDocument:\nPromotion Agreement\n\nPageMaster Corporation\n\nGo Call, Inc.\n\nMarch 12,1999\n\n"Go Call"\n\nThis promotion shall begin on June 1,1999 and shall terminate June 1, 2000 (herein "Term")\n\nThis promotion shall begin on June 1,1999 and shall terminate June 1, 2000 (herein "Term")\n\nThis term shall be extended for a 1 year period provided 3000 pagers per month are distributed to Purchase customers.\n\nPageMaster Corporation shall provide a minimum of 100,000 up to 500,000 pagers for the fulfillment of this promotion to all Purchase Customers who prepay their annual airtime.<|im_end|>\n<|im_start|>assistant\n{\n  "contract_type": "other",\n  "parties": [\n

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [11]:
from trl import SFTTrainer, SFTConfig

import os
os.environ['CC'] = 'gcc'
os.environ['CXX'] = 'g++'

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        output_dir = "outputs_contract_qwen3_4b_demo",
        dataset_text_field = "text",
        max_length = 1024,
        per_device_train_batch_size = 4, # Increased for multi-GPU efficiency
        per_device_eval_batch_size = 4,  # Increased for multi-GPU efficiency
        gradient_accumulation_steps = 1, # Adjusted down as effective batch size is naturally larger with multiple GPUs
        warmup_ratio = 0.1,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        weight_decay = 0.01,
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 50,
        save_steps = 100,
        max_steps = -1,
        optim = "adamw_8bit",
        lr_scheduler_type = "cosine",
        packing = False,
        fp16 = not bf16,
        bf16 = bf16,
        seed = 3407,
        report_to = "none",
        ddp_find_unused_parameters = False, # Best practice for DDP multi-GPU
    ),
)

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [12]:
from unsloth.chat_templates import train_on_responses_only

train_before = len(trainer.train_dataset)
eval_before = len(trainer.eval_dataset)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

train_after = len(trainer.train_dataset)
eval_after = len(trainer.eval_dataset)
removed_fraction = (train_before - train_after) / train_before if train_before else 0.0

print({
    "train_before": train_before,
    "train_after": train_after,
    "eval_before": eval_before,
    "eval_after": eval_after,
    "train_removed_fraction": round(removed_fraction, 4),
})

if train_after == 0:
    raise ValueError("response-only filtering removed every training row.")
if removed_fraction > 0.10:
    raise ValueError(f"Too many rows removed: {removed_fraction:.4f}")

{'train_before': 600, 'train_after': 600, 'eval_before': 80, 'eval_after': 80, 'train_removed_fraction': 0.0}


Let's verify masking the instruction part is done! Let's print the 100th row again.

In [13]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])[:2000]

'<|im_start|>system\nYou are a contract analysis assistant.\nRead the provided agreement and return only valid JSON that matches the contract schema exactly.\nDo not invent clauses, dates, obligations, or penalties.<|im_end|>\n<|im_start|>user\nMode: contract\nTask: extract structured data into the canonical JSON schema.\n\nDocument:\nPromotion Agreement\n\nPageMaster Corporation\n\nGo Call, Inc.\n\nMarch 12,1999\n\n"Go Call"\n\nThis promotion shall begin on June 1,1999 and shall terminate June 1, 2000 (herein "Term")\n\nThis promotion shall begin on June 1,1999 and shall terminate June 1, 2000 (herein "Term")\n\nThis term shall be extended for a 1 year period provided 3000 pagers per month are distributed to Purchase customers.\n\nPageMaster Corporation shall provide a minimum of 100,000 up to 500,000 pagers for the fulfillment of this promotion to all Purchase Customers who prepay their annual airtime.<|im_end|>\n<|im_start|>assistant\n{\n  "contract_type": "other",\n  "parties": [\n

Now let's print the masked out example - you should see only the answer is present:

In [14]:
tokenizer.decode(
    [tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]
).replace(tokenizer.pad_token, " ")[:2000]

'                                                                                                                                                                                                                                   {\n  "contract_type": "other",\n  "parties": [\n    "Go Call"\n  ],\n  "effective_date": "June 1, 2000",\n  "termination_date": "June 1, 2000",\n  "renewal_terms": [\n    "This term shall be extended for a 1 year period provided 3000 pagers per month are distributed to Purchase customers."\n  ],\n  "payment_terms": [\n    "PageMaster Corporation shall provide a minimum of 100,000 up to 500,000 pagers for the fulfillment of this promotion to all Purchase Customers who prepay their annual airtime."\n  ],\n  "key_obligations": [\n    {\n      "party": "unspecified",\n      "obligation": "PageMaster Corporation shall provide a minimum of 100,000 up to 500,000 pagers for the fulfillment of this promotion to all Purchase Customers who prepay their annual airtime.",\n 

In [15]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.609 GB.
3.816 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [16]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 600 | Num Epochs = 1 | Total steps = 150
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 16,515,072 of 4,038,983,168 (0.41% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,0.081200,0.076648
100,0.047600,0.049171
150,0.038000,0.044658


In [17]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

800.7912 seconds used for training.
13.35 minutes used for training.
Peak reserved memory = 9.543 GB.
Peak reserved memory for training = 5.727 GB.
Peak reserved memory % of max memory = 65.323 %.
Peak reserved memory for training % of max memory = 39.202 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Qwen-3` team, the recommended settings for instruct inference are `temperature = 0.7, top_p = 0.8, top_k = 20`

For reasoning chat based inference, `temperature = 0.6, top_p = 0.95, top_k = 20`

In [18]:
messages = [{
    "role": "user",
    "content": """Mode: contract
Task: extract structured data into the canonical JSON schema.

Document:
Master Services Agreement between Klavora Technologies, Inc. and Northwind Systems LLC. Effective date: 2026-01-20. The initial term is 12 months and renews automatically for successive 12-month terms unless either party gives 30 days written notice before renewal. Klavora will pay undisputed invoices within 30 days of receipt. Northwind must maintain service availability of 99.5 percent monthly and provide critical incident response within 4 hours. Each party must protect the other party's confidential information and use it only to perform under this agreement. Klavora may terminate for material breach if the breach is not cured within 15 days after written notice."""
}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = False,
)

inputs = tokenizer(text, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 192,
    do_sample = False,
    eos_token_id = tokenizer.eos_token_id,
    use_cache = True,
)

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print(raw_output)


{
  "task": "contract",
  "document": "Master Services Agreement between Klavora Technologies, Inc. and Northwind Systems LLC. Effective date: 2026-01-20. The initial term is 12 months and renews automatically for successive 12-month terms unless either party gives 30 days written notice before renewal. Klavora will pay undisputed invoices within 30 days of receipt. Northwind must maintain service availability of 99.5 percent monthly and provide critical incident response within 4 hours. Each party must protect the other party's confidential information and use it only to perform under this agreement. Klavora may terminate for material breach if the breach is not cured within 15 days after written notice.",
  "contract_type": "other",
  "parties": [
    "Master Services Agreement between Klavora Technologies, Inc. and Northwind Systems LLC


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [19]:
model.save_pretrained("contract_extract_qwen3_4b_demo")
tokenizer.save_pretrained("contract_extract_qwen3_4b_demo")

# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

('contract_extract_qwen3_4b_demo/tokenizer_config.json',
 'contract_extract_qwen3_4b_demo/special_tokens_map.json',
 'contract_extract_qwen3_4b_demo/chat_template.jinja',
 'contract_extract_qwen3_4b_demo/vocab.json',
 'contract_extract_qwen3_4b_demo/merges.txt',
 'contract_extract_qwen3_4b_demo/added_tokens.json',
 'contract_extract_qwen3_4b_demo/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [20]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [21]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False:
    model.save_pretrained_merged("qwen_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/qwen_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False: # Pushing to HF Hub
    model.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [22]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

Now, use the `qwen_finetune.Q8_0.gguf` file or `qwen_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

In [24]:
import json
from pathlib import Path

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

test_rows = read_jsonl("content/test.jsonl")
len(test_rows)


80

In [42]:
def generate_raw_output(user_prompt, max_new_tokens=1500):
    messages = [
        {"role": "system", "content": "You are an expert contract parser. You must strictly output ONLY valid JSON according to the requested schema. Do not output conversational text or explanations."},
        {"role": "user", "content": user_prompt}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True, # Must add for generation
        return_tensors = "pt",
    ).to("cuda")
    outputs = model.generate(
        input_ids = inputs,
        attention_mask = torch.ones_like(inputs),
        max_new_tokens = max_new_tokens,
        use_cache = True,
        do_sample = False, # Greedy decoding for structured output
    )
    generated_tokens = outputs[0][inputs.shape[1]:]
    raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return raw_output

In [43]:
contract_predictions = []

for i, row in enumerate(test_rows):
    raw_output = generate_raw_output(row["messages"][1]["content"])
    contract_predictions.append({
        "example_id": row["example_id"],
        "raw_output": raw_output,
    })

    if (i + 1) % 10 == 0:
        print(f"done {i + 1}/{len(test_rows)}")


done 10/80
done 20/80
done 30/80
done 40/80
done 50/80
done 60/80
done 70/80
done 80/80


In [44]:
with open("content/contract_qwen_demo.jsonl", "w", encoding="utf-8") as f:
    for row in contract_predictions:
        f.write(json.dumps(row, ensure_ascii=True) + "\n")

print("saved", "content/contract_qwen_demo.jsonl")


saved content/contract_qwen_demo.jsonl


In [45]:
benchmark_rows = read_jsonl("content/contract_demo_benchmark.jsonl")
len(benchmark_rows)


12

In [46]:
benchmark_predictions = []

for i, row in enumerate(benchmark_rows):
    raw_output = generate_raw_output(row["user_prompt"])
    benchmark_predictions.append({
        "example_id": row["example_id"],
        "raw_output": raw_output,
    })

    print(f"done {i + 1}/{len(benchmark_rows)}")


done 1/12
done 2/12
done 3/12
done 4/12
done 5/12
done 6/12
done 7/12
done 8/12
done 9/12
done 10/12
done 11/12
done 12/12


In [47]:
with open("content/contract_qwen_demo_benchmark.jsonl", "w", encoding="utf-8") as f:
    for row in benchmark_predictions:
        f.write(json.dumps(row, ensure_ascii=True) + "\n")

print("saved", "content/contract_qwen_demo_benchmark.jsonl")


saved content/contract_qwen_demo_benchmark.jsonl


In [49]:
from IPython.display import FileLink, display

# Create links to download the files directly from the Jupyter interface
display(FileLink("content/contract_qwen_demo.jsonl"))
display(FileLink("content/contract_qwen_demo_benchmark.jsonl"))
display(FileLink("Qwen3_(4B)_Instruct-v2.ipynb"))
#display(FileLink("/content/evaluation_reports/report.md"))
#display(FileLink("/content/evaluation_reports/report.json"))



/dli/task/content/contract_qwen_demo.jsonl

/dli/task/content/contract_qwen_demo_benchmark.jsonl

/dli/task/Qwen3_(4B)_Instruct-v2.ipynb

In [ ]:
!python3 scripts/evaluate_extraction.py \
    --gold /content/test.jsonl \
    --system qwen3_4b=/content/contract_qwen_demo.jsonl \
    --benchmark-predictions qwen3_4b=/content/contract_qwen_demo_benchmark.jsonl \
    --output-dir /content/evaluation_reports